There is a bug in `pytorch` and you need to make sure that in the conda environment the following variable is set:
```
conda env config vars set KMP_DUPLICATE_LIB_OK=TRUE
```
You can check that the variable is set by running:
```
conda env config vars list
```

In [1]:
from collections import defaultdict

import torch
import torchrl
import tensordict

from envs.mh5robotenv import MH5RobotEnv

from tensordict.nn import  TensorDictModule
from tensordict.nn.distributions import NormalParamExtractor

from torch import nn

from torchrl.collectors import SyncDataCollector
from torchrl.data.replay_buffers import ReplayBuffer
from torchrl.data.replay_buffers.samplers import SamplerWithoutReplacement
from torchrl.data.replay_buffers.storages import LazyTensorStorage
from torchrl.envs import GymEnv, TransformedEnv, Compose, ObservationNorm, DoubleToFloat, StepCounter
from torchrl.envs.utils import check_env_specs, set_exploration_type, ExplorationType

from torchrl.modules import ProbabilisticActor, TanhNormal, ValueOperator
from torchrl.objectives import ClipPPOLoss
from torchrl.objectives.value import GAE

from tqdm import tqdm


In [2]:
print(f"torch version: {torch.__version__}")
print(f"torchrl version: {torchrl.__version__}")
print(f"tensordict version: {tensordict.__version__}")

torch version: 2.9.0
torchrl version: 0.0.0+unknown
tensordict version: 0.10.0


In [3]:
device = torch.device("cpu")
# if torch.backends.mps.is_available():
#     device = torch.device("mps")
# if torch.cuda.is_available():
#     device = torch.device("cuda")
print(device)

cpu


In [4]:
base_env = GymEnv("MH5Robot-v8", device=device)
base_env._env.env.env.model.opt.timestep = 0.0005    # to improve stability

In [5]:
env = TransformedEnv(
    base_env,
    Compose(
        ObservationNorm(in_keys=['observation']),
        DoubleToFloat(),
        StepCounter(),
    ),
)

In [6]:
env.transform[0].init_stats(num_iter=1000, reduce_dim=0, cat_dim=0)
print("normalization constant shape:", env.transform[0].loc.shape)

normalization constant shape: torch.Size([633])


In [7]:
print("observation_spec:", env.observation_spec)
print("reward_spec:", env.reward_spec)
print("input_spec:", env.input_spec)
print("action_spec (as defined by input_spec):", env.action_spec)

observation_spec: Composite(
    observation: UnboundedContinuous(
        shape=torch.Size([633]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([633]), device=cpu, dtype=torch.float32, contiguous=True),
            high=Tensor(shape=torch.Size([633]), device=cpu, dtype=torch.float32, contiguous=True)),
        device=cpu,
        dtype=torch.float32,
        domain=continuous),
    step_count: BoundedDiscrete(
        shape=torch.Size([1]),
        space=ContinuousBox(
            low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, contiguous=True),
            high=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, contiguous=True)),
        device=cpu,
        dtype=torch.int64,
        domain=discrete),
    device=cpu,
    shape=torch.Size([]),
    data_cls=None)
reward_spec: UnboundedContinuous(
    shape=torch.Size([1]),
    space=ContinuousBox(
        low=Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.float32, contiguous=Tru

In [8]:
check_env_specs(env)

2025-10-20 17:24:36,009 [torchrl][INFO]    check_env_specs succeeded! [END]


In [9]:
rollout = env.rollout(3)
print("rollout of three steps:", rollout)
print("Shape of the rollout TensorDict:", rollout.batch_size)

rollout of three steps: TensorDict(
    fields={
        action: Tensor(shape=torch.Size([3, 24]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
        next: TensorDict(
            fields={
                done: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                observation: Tensor(shape=torch.Size([3, 633]), device=cpu, dtype=torch.float32, is_shared=False),
                reward: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.float32, is_shared=False),
                step_count: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.int64, is_shared=False),
                terminated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False),
                truncated: Tensor(shape=torch.Size([3, 1]), device=cpu, dtype=torch.bool, is_shared=False)},
            batch_size=torch.Size([3]),
            de

In [10]:
config = {
    'num_cells': 256,
    'frames_per_batch': 1_000,
    'total_frames': 2_000_000,
    'gamma': 0.99,
    'lmbda': 0.95,
    'clip_epsilon': 0.2,
    'entropy_eps': 1e-4,
    'lr': 3e-4,
    'num_epochs': 10,
    'sub_batch_size': 64,
    'max_grad_norm': 1.0,
}

In [11]:
actor_net = nn.Sequential(
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(2 * env.action_spec.shape[-1], device=device),
    NormalParamExtractor(),
)

policy_module = TensorDictModule(actor_net, in_keys=['observation'], out_keys=['loc', 'scale'])

policy_module = ProbabilisticActor(
    module=policy_module,
    spec=env.action_spec,
    in_keys=['loc', 'scale'],
    distribution_class=TanhNormal,
    distribution_kwargs={
        'low': env.action_spec_unbatched.space.low,
        'high': env.action_spec_unbatched.space.high,
    },
    return_log_prob=True,
)

In [12]:
value_net = nn.Sequential(
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(config['num_cells'], device=device),
    nn.Tanh(),
    nn.LazyLinear(1, device=device)
)

value_module = ValueOperator(
    module=value_net,
    in_keys=['observation'],
)

In [13]:
# we need to do this to "initialize" the Lazy modules
print("Running policy:", policy_module(env.reset()))
print("Running value:", value_module(env.reset()))

Running policy: TensorDict(
    fields={
        action: Tensor(shape=torch.Size([24]), device=cpu, dtype=torch.float32, is_shared=False),
        action_log_prob: Tensor(shape=torch.Size([]), device=cpu, dtype=torch.float32, is_shared=False),
        done: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        loc: Tensor(shape=torch.Size([24]), device=cpu, dtype=torch.float32, is_shared=False),
        observation: Tensor(shape=torch.Size([633]), device=cpu, dtype=torch.float32, is_shared=False),
        scale: Tensor(shape=torch.Size([24]), device=cpu, dtype=torch.float32, is_shared=False),
        step_count: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.int64, is_shared=False),
        terminated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False),
        truncated: Tensor(shape=torch.Size([1]), device=cpu, dtype=torch.bool, is_shared=False)},
    batch_size=torch.Size([]),
    device=cpu,
    is_shared=False)
Runni

In [14]:
collector = SyncDataCollector(
    create_env_fn=env,
    policy=policy_module,
    frames_per_batch=config['frames_per_batch'],
    total_frames=config['total_frames'],
    split_trajs=False,
    device=device,
    use_buffers=False,  # https://github.com/pytorch/rl/issues/3066#issuecomment-3077398138
)

In [15]:
replay_buffer = ReplayBuffer(
    storage=LazyTensorStorage(max_size=config['frames_per_batch']),
    sampler=SamplerWithoutReplacement(),
)

In [16]:
advantage_module = GAE(
    gamma=config['gamma'],
    lmbda=config['lmbda'],
    value_network=value_module,
    average_gae=True,
)

loss_module = ClipPPOLoss(
    actor_network=policy_module,
    critic_network=value_module,
    clip_epsilon=config['clip_epsilon'],
    entropy_bonus=bool(config['entropy_eps']),
    entropy_coeff=config['entropy_eps'],
    critic_coeff=1.0,
    loss_critic_type='smooth_l1',
)

optim = torch.optim.Adam(loss_module.parameters(), config['lr'])
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer=optim,
    T_max=(config['total_frames'] // config['frames_per_batch']),
    eta_min=0.0
)

In [17]:
logs = defaultdict(list)
pbar = tqdm(total=config['total_frames'])
eval_str = ""

# collector.verbose = True

for i, tensordict_data in enumerate(collector):
    for _ in range (config['num_epochs']):
        advantage_module(tensordict_data)
        data_view = tensordict_data.reshape(-1)
        replay_buffer.extend(data_view.cpu())
        for _ in range(config['frames_per_batch'] // config['sub_batch_size']):
            subdata = replay_buffer.sample(config['sub_batch_size'])
            loss_vals = loss_module(subdata.to(device))
            loss_value = (
                loss_vals['loss_objective']
                + loss_vals['loss_critic']
                + loss_vals['loss_entropy']
            )
            loss_value.backward()
            torch.nn.utils.clip_grad_norm_(loss_module.parameters(), config['max_grad_norm'])
            optim.step()
            optim.zero_grad()

    logs['reward'].append(tensordict_data['next', 'reward'].mean().item())
    pbar.update(tensordict_data.numel())
    cum_reward_str = f"average reward={logs['reward'][-1]:4.4f} (init={logs['reward'][0]:4.4f})"
    logs['step_count'].append(tensordict_data['step_count'].max().item())
    stepcount_str = f"step count (max): {logs['step_count'][-1]}"
    logs['lr'].append(optim.param_groups[0]['lr'])
    lr_str = f"lr policy: {logs['lr'][-1]:4.4f}"

    if i % 10 == 0:
        with set_exploration_type(ExplorationType.DETERMINISTIC), torch.no_grad():
            eval_rollout = env.rollout(1000, policy_module)
            logs['eval reward'].append(eval_rollout['next', 'reward'].mean().item())
            logs["eval reward (sum)"].append(eval_rollout["next", "reward"].sum().item())
            logs["eval step_count"].append(eval_rollout["step_count"].max().item())
            eval_str = (
                f"eval cumulative reward: {logs['eval reward (sum)'][-1]:4.4f} "
                f"(init: {logs['eval reward (sum)'][0]:4.4f}) "
                f"eval step-count: {logs['eval step_count'][-1]}"
            )
            del eval_rollout

    pbar.set_description(", ".join([eval_str, cum_reward_str, stepcount_str, lr_str]))

    scheduler.step()


eval cumulative reward: 593.9431 (init: 668.9050) eval step-count: 269, average reward=1.2712 (init=0.9846), step count (max): 727, lr policy: 0.0003:  10%|█         | 203000/2000000 [07:33<1:07:38, 442.73it/s] 

eval cumulative reward: 593.9431 (init: 668.9050) eval step-count: 269, average reward= nan (init=0.9846), step count (max): 710, lr policy: 0.0003:  10%|█         | 204000/2000000 [07:35<1:06:53, 447.54it/s]  

eval cumulative reward: 593.9431 (init: 668.9050) eval step-count: 269, average reward= nan (init=0.9846), step count (max): 999, lr policy: 0.0003:  10%|█         | 205000/2000000 [07:37<1:06:19, 451.11it/s]

eval cumulative reward: 593.9431 (init: 668.9050) eval step-count: 269, average reward= nan (init=0.9846), step count (max): 999, lr policy: 0.0003:  10%|█         | 206000/2000000 [07:39<1:05:31, 456.30it/s]

eval cumulative reward: 593.9431 (init: 668.9050) eval step-count: 269, average reward= nan (init=0.9846), step count (max): 999, lr policy: 0.0003:  10%|█         | 207000/2000000 [07:41<1:05:00, 459.70it/s]

...

eval cumulative reward:  nan (init: 668.9050) eval step-count: 999, average reward= nan (init=0.9846), step count (max): 999, lr policy: 0.0003:  11%|█         | 221000/2000000 [08:12<1:04:52, 456.98it/s]

eval cumulative reward:  nan (init: 668.9050) eval step-count: 999, average reward= nan (init=0.9846), step count (max): 999, lr policy: 0.0003:  11%|█         | 221000/2000000 [08:14<1:04:52, 456.98it/s]

eval cumulative reward:  nan (init: 668.9050) eval step-count: 999, average reward= nan (init=0.9846), step count (max): 999, lr policy: 0.0003:  11%|█         | 222000/2000000 [08:16<1:14:45, 396.41it/s]

eval cumulative reward:  nan (init: 668.9050) eval step-count: 999, average reward= nan (init=0.9846), step count (max): 999, lr policy: 0.0003:  11%|█         | 223000/2000000 [08:18<1:12:07, 410.59it/s]

KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 10))
plt.subplot(2, 2, 1)
plt.plot(logs["reward"])
plt.title("training rewards (average)")
plt.subplot(2, 2, 2)
plt.plot(logs["step_count"])
plt.title("Max step count (training)")
plt.subplot(2, 2, 3)
plt.plot(logs["eval reward (sum)"])
plt.title("Return (test)")
plt.subplot(2, 2, 4)
plt.plot(logs["eval step_count"])
plt.title("Max step count (test)")
plt.show()